# Train and evaluate on Colab

Run the cells in order. The cache made by `prepare_data.py` on your computer
has to be on Google Drive first (see `docs/test_plan.md`). Runs are saved on
Drive, so when the session is cut you can open the notebook again, run every
cell and training continues from the last checkpoint.

Runtime > Change runtime type > T4 GPU.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# change these if you keep the files somewhere else on Drive
DRIVE_DIR = "/content/drive/MyDrive/mri-diffusion"
RUN_NAME = "pilot"         # "full" for the run with every subject
TRAIN_STEPS = 20000        # pilot, 40000 to 60000 for the full run

import os
os.environ["MRI_CACHE_DIR"] = "/content/cache"          # local copy, faster than reading Drive
os.environ["MRI_RUNS_DIR"] = f"{DRIVE_DIR}/runs"         # checkpoints survive a cut session
os.environ["MRI_RUN_NAME"] = RUN_NAME
os.environ["MRI_TRAIN_STEPS"] = str(TRAIN_STEPS)

In [ ]:
%cd /content
!git -C brain-mri-diffusion-sr pull || git clone -b refactor https://github.com/acavieira/brain-mri-diffusion-sr.git
%cd /content/brain-mri-diffusion-sr
!pip install -q nibabel opencv-python-headless scikit-image

In [ ]:
!mkdir -p /content/cache && cp -n {DRIVE_DIR}/cache/*.npz {DRIVE_DIR}/cache/splits.csv /content/cache/
!ls -lh /content/cache
!python -m pytest -q

## Training

Prints the loss and the time per step every 100 steps, and the validation
scores every `VALIDATE_EVERY` steps. Settings are in
`src/mri_diffusion/config.py`. To change one for this session only, edit the
file in the Colab file browser before running the cell.

In [ ]:
!python scripts/train.py

## Evaluation on the test subjects

In [ ]:
!python scripts/evaluate.py

In [ ]:
import pandas as pd
from IPython.display import Image, display

run = f"{os.environ['MRI_RUNS_DIR']}/{RUN_NAME}"
display(pd.read_csv(f"{run}/test/summary.csv"))
log = pd.read_csv(f"{run}/log.csv")
log.plot(x="step", y=["train_loss", "val_loss"], logy=True)
log.plot(x="step", y="val_psnr_brain")
for name in sorted(os.listdir(f"{run}/test/examples"))[:3]:
    display(Image(f"{run}/test/examples/{name}"))